In [1]:
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv

load_dotenv()
from langchain.chat_models import init_chat_model

api_key= os.getenv("GROQ_API_KEY")

llm = ChatGroq(model = "llama-3.1-8b-instant", api_key = api_key)

In [4]:
from pydantic import BaseModel, Field

In [5]:
class ContactInfo(BaseModel):
    name: str = Field(..., description="Full name of the person")
    email: str = Field(..., description="Email address")
    phone: str = Field(..., description="Phone number if mentioned, else 'N/A'")

In [6]:
from langchain_core.output_parsers import PydanticOutputParser

parser = PydanticOutputParser(pydantic_object = ContactInfo)

In [7]:
parser

PydanticOutputParser(pydantic_object=<class '__main__.ContactInfo'>)

# For Better Understanding of How this structured output block works in Prompt

In [9]:
parser.get_format_instructions()

'The output should be formatted as a JSON instance that conforms to the JSON schema below.\n\nAs an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}\nthe object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.\n\nHere is the output schema:\n```\n{"properties": {"name": {"description": "Full name of the person", "title": "Name", "type": "string"}, "email": {"description": "Email address", "title": "Email", "type": "string"}, "phone": {"description": "Phone number if mentioned, else \'N/A\'", "title": "Phone", "type": "string"}}, "required": ["name", "email", "phone"]}\n```'

## I guess Pydantic Output Instruction itself might not be enough in many cases and you need to define it further in Prompt yourself!

In [12]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an information extraction assistant."),
    ("human", "Extract contact details from the text."),
    ("human", "{text}"),
    ("system", "Format your response as:\n{format_instructions}")
])

In [13]:
prompt = prompt.partial(format_instructions=parser.get_format_instructions())

## Basic Langchain Chain

In [14]:
chain = prompt | llm | parser

In [15]:
text = "You can reach John Doe at john.doe@example.com or call him at +1-202-555-0147."
result = chain.invoke({"text": text})

In [17]:
print(result)

name='John Doe' email='john.doe@example.com' phone='+1-202-555-0147'


In [19]:
print(f"Name: {result.name} \nEmail: {result.email} \nPhone No: {result.phone}")

Name: John Doe 
Email: john.doe@example.com 
Phone No: +1-202-555-0147


### Getting Structured Output as expected!